In [1]:
import os

In [2]:
import json

In [3]:
import requests
import numpy as np

In [4]:
import cv2

In [5]:
from flask import Flask, request, redirect, url_for, render_template_string
from tensorflow.keras.models import load_model

In [6]:
from werkzeug.utils import secure_filename
import webbrowser

In [7]:
import threading

In [8]:
from PIL import Image

In [9]:
import base64
from io import BytesIO

In [10]:
from markupsafe import Markup

In [11]:
import re

In [12]:
# Load model
model = load_model("D:\\Projects\\MAJOR PROJECT\\plant_disease_model.h5")

# Flask setup
app = Flask(__name__)
app.config['UPLOAD_FOLDER'] = 'static/uploaded_images'
os.makedirs(app.config['UPLOAD_FOLDER'], exist_ok=True)

# Class names
class_names = [ 'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
    'Blueberry___healthy', 'Cherry_(including_sour)___healthy', 'Cherry_(including_sour)___Powdery_mildew',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_',
    'Corn_(maize)___healthy', 'Corn_(maize)___Northern_Leaf_Blight', 'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)', 'Grape___healthy', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
    'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy',
    'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight',
    'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___healthy',
    'Strawberry___Leaf_scorch', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___healthy',
    'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_mosaic_virus',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus'
]

# OpenRouter API Key
API_KEY = "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"

def get_ai_suggestion(disease_name):
    try:
        # Skip API call for healthy plants
        if "healthy" in disease_name.lower():
            return {
                "overview": "This plant appears to be healthy!",
                "prevention": Markup("<ul><li>Water appropriately</li><li>Ensure proper sunlight exposure</li><li>Fertilize as needed</li><li>Monitor regularly for early signs of issues</li></ul>"),
                "treatment": "No treatment needed as the plant is healthy. Continue with regular care practices."
            }
        
        headers = {
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
            "HTTP-Referer": "http://localhost",
            "X-Title": "Plant Disease Classifier"
        }

        # Improve prompt to ensure consistent formatting
        data = {
            "model": "openai/gpt-4o",
            "messages": [
                {"role": "system", "content": "You are a plant disease expert. Provide advice in a structured format with three sections: Overview, Prevention, and Treatment. Each section should be clearly labeled. For Prevention and Treatment, provide information in HTML bullet point format using <ul><li> tags. Keep each section concise and actionable."},
                {"role": "user", "content": f"Provide prevention and treatment information for the plant disease: {disease_name}. Format your response with three clearly labeled sections: Overview, Prevention, and Treatment. For Prevention and Treatment, use HTML bullet points with <ul><li> format."}
            ],
            "max_tokens": 500
        }

        response = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, data=json.dumps(data))
        result = response.json()
        
        if "choices" in result and result["choices"]:
            content = result["choices"][0]["message"]["content"]
            
            # Improved section extraction
            sections = {
                "overview": "",
                "prevention": "",
                "treatment": ""
            }
            
            # Split content by section headers
            pattern = re.compile(r'(overview|prevention|treatment):', re.IGNORECASE)
            parts = pattern.split(content)
            
            # Process the parts
            current_section = None
            for part in parts:
                part = part.strip()
                lower_part = part.lower()
                
                if lower_part in ["overview", "prevention", "treatment"]:
                    current_section = lower_part
                elif current_section and part:
                    # Format prevention and treatment as HTML if not already
                    if current_section in ["prevention", "treatment"] and not part.strip().startswith("<ul>"):
                        # Convert plain text bullets to HTML
                        if re.search(r'^\s*[-•*]\s', part, re.MULTILINE):
                            lines = part.split('\n')
                            html_content = "<ul>"
                            for line in lines:
                                line = line.strip()
                                if re.match(r'^[-•*]\s', line):
                                    item_content = re.sub(r'^[-•*]\s', '', line)
                                    html_content += f"<li>{item_content}</li>"
                            html_content += "</ul>"
                            sections[current_section] = Markup(html_content)
                        else:
                            # No bullet points found, wrap the entire content in ul/li
                            html_content = f"<ul><li>{part}</li></ul>"
                            sections[current_section] = Markup(html_content)
                    else:
                        sections[current_section] = Markup(part.strip())
            
            # If parsing fails, use the entire content as overview
            if not sections["overview"] and not sections["prevention"] and not sections["treatment"]:
                sections["overview"] = Markup(content)
                sections["prevention"] = Markup("<ul><li>Please consult a plant specialist for prevention methods.</li></ul>")
                sections["treatment"] = Markup("<ul><li>Please consult a plant specialist for treatment options.</li></ul>")
            
            # Ensure all sections have content
            for section in sections:
                if not sections[section]:
                    if section == "overview":
                        sections[section] = Markup("No overview information available.")
                    else:
                        sections[section] = Markup(f"<ul><li>No {section} information available.</li></ul>")
            
            return sections
        else:
            return {
                "overview": "Unable to get suggestion from AI service.",
                "prevention": Markup("<ul><li>Please try again later or consult a plant specialist.</li></ul>"),
                "treatment": Markup("<ul><li>No treatment information available at this moment.</li></ul>")
            }
    except Exception as e:
        return {
            "overview": f"Error: {e}",
            "prevention": Markup("<ul><li>Please try again later or check your internet connection.</li></ul>"),
            "treatment": Markup("<ul><li>No treatment information available due to an error.</li></ul>")
        }

HTML_TEMPLATE = '''
<!DOCTYPE html>
<html>
<head>
    <title>🌿 Plant Disease Classifier</title>
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        body { 
            font-family: 'Segoe UI', sans-serif; 
            margin: 0; 
            padding: 0; 
            background: linear-gradient(to bottom right, #f1f8e9, #dcedc8); 
            color: #333; 
            text-align: center;
            padding-bottom: 40px;
        }
        .container {
            max-width: 800px;
            margin: 0 auto;
            padding: 20px;
        }
        h1 { margin-top: 30px; font-size: 2.5em; color: #388e3c; }
        h3 { color: #2e7d32; }
        form { margin-top: 30px; }
        button { 
            margin: 10px; 
            padding: 10px 20px; 
            font-size: 1em; 
            background-color: #66bb6a; 
            color: white; 
            border: none; 
            border-radius: 8px; 
            cursor: pointer;
            transition: background-color 0.3s;
        }
        button:hover { background-color: #4caf50; }
        input[type="file"] { font-size: 1em; }
        video, canvas { display: none; margin: 10px auto; max-width: 100%; }
        img { max-width: 300px; margin: 20px auto; border: 2px solid #ccc; border-radius: 10px; display: block; }
        .result { 
            margin: 30px auto; 
            padding: 20px; 
            background: #e8f5e9; 
            border-radius: 12px; 
            box-shadow: 0px 0px 10px rgba(0,0,0,0.1);
            text-align: left;
            max-width: 80%;
        }
        .result h2 { 
            color: #2e7d32; 
            text-align: center;
            border-bottom: 1px solid #a5d6a7;
            padding-bottom: 10px;
        }
        .result-section {
            margin: 15px 0;
            padding: 10px;
            background: rgba(255, 255, 255, 0.7);
            border-radius: 8px;
        }
        .result-section h3 {
            margin-top: 0;
            color: #388e3c;
            border-bottom: 1px solid #c8e6c9;
            padding-bottom: 5px;
        }
        .result-section ul {
            margin-top: 10px;
            padding-left: 25px;
            text-align: left; 
        }
        .result-section li {
            margin-bottom: 5px;
        }
        #loader { 
            display: none; 
            margin: 20px auto; 
            text-align: center;
        }
        #loader img { width: 100px; }
        
        /* Responsive adjustments */
        @media (max-width: 600px) {
            h1 { font-size: 2em; }
            .result { padding: 15px; max-width: 90%; }
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>🌿 Plant Disease Classifier</h1>
        <form method="POST" enctype="multipart/form-data" id="upload-form">
            <input type="file" name="file" accept="image/*" id="file-input" required onchange="previewImage(event)">
            <br><br>
            <button type="button" id="camera-btn" onclick="startCamera()">📷 Use Webcam</button>
            <button type="button" id="capture-btn" style="display: none;" onclick="captureImage()">📸 Take Photo</button>
            <button type="submit" id="submit-btn">🚀 Classify</button>
            <br>
            <video id="video" autoplay></video>
            <canvas id="canvas" width="300" height="225"></canvas>
            <input type="hidden" name="webcam_image" id="webcam-image">
            <div id="loader">
                <p>Processing image...</p>
                <img src="https://i.gifer.com/ZZ5H.gif" alt="Loading...">
            </div>
        </form>

        {% if image_url %}
            <h3>Image Preview:</h3>
            <img src="{{ image_url }}" id="preview-image" alt="Plant Image">
        {% endif %}

        {% if prediction %}
            <div class="result">
                <h2>Analysis Results</h2>
                <div class="result-section">
                    <h3>🔍 Identified Issue</h3>
                    <p>{{ prediction.replace('___', ' - ').replace('_', ' ') }}</p>
                </div>
                
                <div class="result-section">
                    <h3>📋 Overview</h3>
                    <p>{{ suggestion.get('overview', 'No overview available.') }}</p>
                </div>
                
                <div class="result-section">
                    <h3>🛡️ Prevention</h3>
                    <div>{{ suggestion.get('prevention', 'No prevention information available.') }}</div>
                </div>
                
                <div class="result-section">
                    <h3>💊 Treatment</h3>
                    <div>{{ suggestion.get('treatment', 'No treatment information available.') }}</div>
                </div>
            </div>
        {% endif %}
    </div>

<script>
    const video = document.getElementById('video');
    const canvas = document.getElementById('canvas');
    const webcamInput = document.getElementById('webcam-image');
    const fileInput = document.getElementById('file-input');
    const form = document.getElementById('upload-form');
    const loader = document.getElementById('loader');
    const cameraBtn = document.getElementById('camera-btn');
    const captureBtn = document.getElementById('capture-btn');
    const submitBtn = document.getElementById('submit-btn');
    
    let stream = null;

    function startCamera() {
        cameraBtn.disabled = true;
        cameraBtn.innerText = "Starting Camera...";
        
        navigator.mediaDevices.getUserMedia({ video: true })
            .then(videoStream => {
                stream = videoStream;
                video.srcObject = stream;
                video.style.display = 'block';
                cameraBtn.style.display = 'none';  // Hide the camera button
                captureBtn.style.display = 'inline-block';  // Show the capture button
            })
            .catch(error => {
                console.error("Error accessing webcam: ", error);
                alert("Could not access webcam. Please check your camera permissions.");
                cameraBtn.disabled = false;
                cameraBtn.innerText = "📷 Use Webcam";
            });
    }

    function captureImage() {
        if (!stream) return;
        
        const context = canvas.getContext('2d');
        context.drawImage(video, 0, 0, canvas.width, canvas.height);
        
        const imageDataURL = canvas.toDataURL('image/jpeg');
        webcamInput.value = imageDataURL;
        fileInput.removeAttribute('required');
        fileInput.value = ''; // Clear file input
        
        // Create or update preview image
        let previewImg = document.getElementById('preview-image');
        if (!previewImg) {
            previewImg = document.createElement('img');
            previewImg.id = 'preview-image';
            previewImg.alt = "Captured Image";
            previewImg.style.display = "block";  // Make sure it's visible
            document.querySelector('.container').appendChild(previewImg);
        }
        previewImg.src = imageDataURL;
        
        // Stop webcam stream
        stream.getTracks().forEach(track => track.stop());
        stream = null;
        video.style.display = 'none';
        
        // Reset the buttons
        cameraBtn.style.display = 'inline-block';
        cameraBtn.disabled = false;
        cameraBtn.innerText = "📷 Use Webcam";
        captureBtn.style.display = 'none';
        
        alert("Image captured successfully! Click 'Classify' to analyze the plant.");
    }

    function previewImage(event) {
        const file = event.target.files[0];
        if (!file) return;
        
        const reader = new FileReader();
        reader.onload = function() {
            let previewImg = document.getElementById('preview-image');
            if (!previewImg) {
                previewImg = document.createElement('img');
                previewImg.id = 'preview-image';
                previewImg.alt = "Selected Image";
                document.querySelector('.container').appendChild(previewImg);
            }
            previewImg.src = reader.result;
            
            // Clear webcam data if file is uploaded
            webcamInput.value = '';
        };
        reader.readAsDataURL(file);
    }

    form.addEventListener("submit", function(event) {
        const fileInput = document.getElementById('file-input');
        const webcamImage = document.getElementById('webcam-image');
        
        // Check if either file or webcam image is provided
        if (!fileInput.files.length && !webcamImage.value) {
            event.preventDefault();
            alert("Please select an image or use the webcam.");
            return false;
        }
        
        loader.style.display = "block";
        submitBtn.disabled = true;
        cameraBtn.disabled = true;
        captureBtn.disabled = true;
    });
</script>
</body>
</html>
'''

@app.route("/", methods=["GET", "POST"])
def index():
    prediction = None
    suggestion = None
    image_url = None

    if request.method == "POST":
        image = None

        if "webcam_image" in request.form and request.form["webcam_image"]:
            try:
                webcam_data = request.form["webcam_image"].split(",")[1]
                image_bytes = base64.b64decode(webcam_data)
                
                # Save the webcam image
                image_path = os.path.join(app.config['UPLOAD_FOLDER'], 'webcam_image.jpg')
                with open(image_path, 'wb') as f:
                    f.write(image_bytes)
                
                # Process image for classification
                pil_image = Image.open(BytesIO(image_bytes)).convert('RGB')
                pil_image = pil_image.resize((150, 150))
                img_array = np.array(pil_image) / 255.0
                image = np.expand_dims(img_array, axis=0)
                
                image_url = url_for("static", filename=f"uploaded_images/webcam_image.jpg")
            except Exception as e:
                print(f"Error processing webcam image: {e}")

        elif "file" in request.files and request.files["file"]:
            try:
                file = request.files["file"]
                if file.filename != '':
                    filename = secure_filename(file.filename)
                    filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
                    file.save(filepath)
                    
                    # Process image for classification
                    img = cv2.imread(filepath)
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
                    img = cv2.resize(img, (150, 150))
                    img = img / 255.0
                    image = np.expand_dims(img, axis=0)
                    
                    image_url = url_for("static", filename=f"uploaded_images/{filename}")
            except Exception as e:
                print(f"Error processing uploaded file: {e}")
        
        if image is not None:
            try:
                # Print shape for debugging
                print(f"Image shape for prediction: {image.shape}")
                
                # Make prediction
                pred = model.predict(image)
                class_index = np.argmax(pred)
                
                print(f"Prediction result: class_index={class_index}, confidence={np.max(pred)}")
                
                if 0 <= class_index < len(class_names):
                    prediction = class_names[class_index]
                    suggestion = get_ai_suggestion(prediction)
                else:
                    prediction = "Unknown Class"
                    suggestion = {
                        "overview": "The image could not be classified into a known plant disease category.",
                        "prevention": Markup("<ul><li>Consider taking a clearer image</li><li>Ensure proper lighting</li><li>Consult with a plant expert</li></ul>"),
                        "treatment": Markup("<ul><li>No specific treatment can be recommended without proper disease identification</li></ul>")
                    }
            except Exception as e:
                print(f"Error during classification: {e}")
                prediction = "Error in Classification"
                suggestion = {
                    "overview": f"An error occurred during classification: {str(e)}",
                    "prevention": Markup("<ul><li>Please try again with a different image</li></ul>"),
                    "treatment": Markup("<ul><li>No treatment information available due to classification error</li></ul>")
                }

    return render_template_string(HTML_TEMPLATE, prediction=prediction, suggestion=suggestion, image_url=image_url)

def run_flask():
    try:
        webbrowser.open("http://127.0.0.1:5000")
        app.run(debug=False, use_reloader=False)
    except Exception as e:
        print(f"Error starting the Flask application: {e}")

# Start app
if __name__ == "__main__":
    threading.Thread(target=run_flask).start()

C:\Users\kumar\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [19/Apr/2025 19:22:46] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Apr/2025 19:22:46] "GET /favicon.ico HTTP/1.1" 404 -


Image shape for prediction: (1, 150, 150, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 315ms/step


INFO:werkzeug:127.0.0.1 - - [19/Apr/2025 19:23:40] "POST / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Apr/2025 19:23:40] "GET /static/uploaded_images/00e909aa-e3ae-4558-9961-336bb0f35db3___JR_FrgE.S_8593_90deg.JPG HTTP/1.1" 200 -


Prediction result: class_index=9, confidence=0.9858608245849609
Image shape for prediction: (1, 150, 150, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step


INFO:werkzeug:127.0.0.1 - - [19/Apr/2025 19:25:28] "POST / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Apr/2025 19:25:28] "GET /static/uploaded_images/webcam_image.jpg HTTP/1.1" 200 -


Prediction result: class_index=9, confidence=0.7279026508331299
Image shape for prediction: (1, 150, 150, 3)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step
Prediction result: class_index=1, confidence=1.0


INFO:werkzeug:127.0.0.1 - - [19/Apr/2025 19:26:11] "POST / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [19/Apr/2025 19:26:11] "GET /static/uploaded_images/0ebea6f4-08e4-4380-86f8-34d854697e32___JR_FrgE.S_2877.JPG HTTP/1.1" 200 -
